# MCP 05 · 部署与调试（Docker / ASGI 生产部署 / MCP Inspector）

前四课我们一直在**本机把 MCP 服务跑起来**：`python 01_服务端.py http` 一按，服务就在
`127.0.0.1:8100` 上等着了。这一课回答两个「跑通之后」的问题：

- **怎么把它搬到服务器上？** —— 开发形态 → ASGI 应用 → Docker 镜像，一条升级路径；
- **怎么在不写客户端代码的前提下确认它是对的？** —— MCP Inspector 点几下，工具 / 资源 / 提示词全看清。

| 概念 | 是什么 | 本课的代码形态 |
|---|---|---|
| 开发部署 | `mcp.run(transport="streamable-http")`，框架替你起一个 uvicorn | `server.py`（课案原文） |
| 生产部署 | `mcp.http_app()` 把 FastMCP 还原成**标准 ASGI 应用**，自己交给 uvicorn | `app.py` + `uvicorn app:app --workers 4` |
| 容器化 | 把服务、依赖、启动命令一起打成一个镜像 | `Dockerfile` + `docker build` / `docker run` |
| MCP Inspector | 官方可视化调试器（前端 6274 + 代理 6277） | `npx @modelcontextprotocol/inspector` |

> **本 notebook 由 `Agent/05_mcp/` 下 3 个脚本合并而成**：
> `10_部署_Docker.py`（课案原版 48 行）、`11_部署_jxsd.py`（完整版 307 行）、
> `12_调试工具_jxsd.py`（完整版 245 行）。

> ⚠️ **本课标 🔴，但 Docker 段落是「只演示、不执行」**：本机 Docker 引擎当前账号
> **无权限访问**，所以第 1 节把 Dockerfile 与 docker 命令**原样打印出来**当作教学内容，
> **不调用任何 docker 命令** —— 也就不会冒出「access denied」这种看着像课程报错的输出。
> 真要构建镜像，把第 1 节的命令拿到有权限的机器上照着敲即可。

**官方文档**
- FastMCP HTTP 部署（`http_app` / uvicorn / Docker）：<https://gofastmcp.com/deployment/http>
- MCP Inspector 官方文档：<https://modelcontextprotocol.io/docs/tools/inspector>
- Inspector 源码仓库：<https://github.com/modelcontextprotocol/inspector>

## 运行条件

| 项 | 说明 |
|---|---|
| 🔴 运行档位 | **需外部服务** —— 讲的是 Docker 部署与 Inspector 调试；本机 Docker 引擎**无权限访问**，Docker 段落**只展示命令、不执行**，失败也不当报错 |
| 依赖 | `fastmcp` / `uvicorn`（venv 已装）；`npx` + `node`（本机已装，用于说明 Inspector） |
| 密钥 | 无 —— 本课不读 `.env`、不连大模型 |
| 前置服务 | **不需要你预先启动任何东西**：第 2 节会在**当前内核进程内**真起一次 ASGI 应用（`127.0.0.1:8024`），连上去调一次工具，跑完立刻关闭 |
| 端口 | `8024`（第 2 节的自检）、`8100`（第 3 节告诉 Inspector 去连的 MCP 服务端） |
| 预计耗时 | 约 15 秒（最慢的是 `npx --version`，首次可能要触网） |

**这一课哪些能真跑、哪些只演示**（照着看，不会误以为课程坏了）：

| 小节 | 能不能实跑 | 说明 |
|---|---|---|
| 1. Dockerfile / docker-compose 打印 | ✅ 能跑 | 只是 `print` 两个多行字符串，**不需要 Docker** |
| 1.2 `docker build` / `docker run` | ❌ 只演示 | 需要 Docker 才能实跑；本机账号无权限，所以只讲命令与逐行含义 |
| 2.1 开发环境 `server.py` | 📖 只展示源码 | `mcp.run()` 会**永久阻塞**内核，notebook 里不执行，只打印原文 |
| 2.2 生产环境 `app.py` | 📖 只展示源码 | 同上，`uvicorn` 命令行是给人敲的 |
| 2.3 生产 Dockerfile | ✅ 能跑（打印） | 与 1.1 同理，只打印 |
| **2.4 ASGI 自检** | ✅ **真跑** | 在当前进程里真起一次 `mcp.http_app()` 造出来的 ASGI 应用并连上去调工具 |
| 3.1 npx / node 检查 | ✅ 真跑 | 真执行一次 `npx --version` |
| 3.2 服务端在不在 | ✅ 真跑 | 探测 `127.0.0.1:8100`；没有服务只打印提示，不报错 |
| 3.3 / 3.4 Inspector 手册 | ✅ 能跑（打印） | 打印课案原文的操作步骤 |
| `npx @modelcontextprotocol/inspector` 本身 | ❌ 不启动 | 它是**前台常驻**进程，跑在 notebook 里会把内核卡死 —— 只打印命令，由你自己开终端跑 |

## 本节地图

一条「从本机脚本到线上镜像」的升级路径，外加一条「不写代码就能验证」的旁路：

```mermaid
graph LR
    A["server.py<br/>mcp.run()<br/>开发：框架代起 uvicorn"] --> B["app.py<br/>mcp.http_app()<br/>生产：标准 ASGI 应用"]
    B --> C["uvicorn app:app<br/>--workers 4<br/>多进程 / 中间件"]
    B --> D["Dockerfile<br/>docker build<br/>docker run"]
    C --> D
    E["MCP Inspector<br/>npx @modelcontextprotocol/inspector<br/>手工探索"] -.->|连 URL/stdio| B
    F["❌ 不会在 notebook 里做的两件事：<br/>执行 docker 命令 / 启动 Inspector"] -.-> D
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 从 | 到 | 靠什么 | 为什么 |
|---|---|---|---|
| 开发形态 | 生产形态 | `mcp.http_app(transport="streamable-http")` | 把 FastMCP 还原成标准 ASGI 应用，才挂得上中间件、开得了多进程 |
| 生产形态 | 多进程 | `uvicorn app:app --workers 4` | `app:app` = 模块名 : 变量名 |
| 生产形态 | 容器 | `docker build` → `docker run -d -p 8000:8000` | 依赖 + 代码 + 启动命令一起固化 |
| 任何形态 | 可视化验证 | Inspector 选 Streamable HTTP，URL 填 `http://<主机>:<端口>/mcp` | 不写客户端代码就能看清工具 / 资源 / 提示词 |
| 本地 .py | 可视化验证 | Inspector 选 STDIO，填 Command + Args | Inspector 自己拉子进程，不占端口 |

**和上下节的衔接**：上一课 `04_权限_JWT认证.ipynb` 给服务端加上了 JWT 鉴权（一个 ASGI 中间件），
本课第 2 节正好解释「为什么有了 `mcp.http_app()` 才能这么挂」；下一课进入
`06_langfuse/`，那里会把「服务跑起来之后怎么观察它」从**手工点界面**升级成**自动埋点**。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

> 本课其实用不到 `config`（不读密钥、不连模型），但这一格仍然保留 —— 一是保持全仓统一，
> 二是它顺便给出了 `NB_DIR` / `WORKDIR` 两个变量：第 3 节讲 Inspector 的 STDIO 方式时，
> 要用 `NB_DIR` 拼出「服务端脚本在哪」，因为 notebook 里**没有 `__file__`**。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

### 预期输出

```text
仓库根： F:\ProGram\Python_Base
临时目录： F:\ProGram\Python_Base\Agent\05_mcp\tmp_nb_work
```

两行都是**绝对路径**，会随你把仓库放在哪而变。要点是：
第一行必须是**仓库根**（`config.py` 所在的那一层）；第二行必须是**本 notebook 目录下的
`tmp_nb_work`** —— 同章并发的 notebook 会在里面各用各的子目录，不互相踩。

### 前置条件自检

本课**不需要密钥、不需要模型、不需要你提前起服务**，这一格只确认两件事：

1. `fastmcp` / `uvicorn` 能不能 import —— 它们决定第 2 节的 ASGI 自检跑不跑得起来；
2. `docker` 可执行文件在不在 —— **只看路径、不调用引擎**（本机账号无权限，
   调了只会拿到 access denied，而那不是课程内容的失败）。

第三件事「端口 `8024` 空不空」**不在这里查**，而是第 2 节在真正起服务之前现查
—— 不空就跳过自检，不硬抢。

自检结果写进 `CAN_RUN_ASGI`，第 2 节用它决定「真跑」还是「打印一句中文说明就跳过」。

In [ ]:
import importlib
import shutil

CAN_RUN_ASGI = True
for _mod in ("fastmcp", "uvicorn"):
    try:
        _m = importlib.import_module(_mod)
        print(f"✅ {_mod} 已安装：{getattr(_m, '__version__', '未知版本')}")
    except Exception as _exc:                      # noqa: BLE001 —— 缺依赖不该让整本 notebook 崩
        CAN_RUN_ASGI = False
        print(f"⚠️  {_mod} 不可用（{type(_exc).__name__}）：第 2 节的 ASGI 自检会跳过，其余说明不受影响。")

DOCKER_BIN = shutil.which("docker")
if DOCKER_BIN:
    # 只证明「这个可执行文件在 PATH 里」，不证明「当前账号有权限访问引擎」——
    # 本课刻意不去调它：本机无权限时会抛 access denied，那属于环境限制，不是课程内容。
    print(f"ℹ️  找到 docker 可执行文件：{DOCKER_BIN}")
    print("   本课只展示 docker 命令，不调用引擎（无权限的机器上会报 access denied）。")
else:
    print("ℹ️  PATH 里没有 docker：第 1 节的命令请在装了 Docker 的机器上执行。")

print("CAN_RUN_ASGI =", CAN_RUN_ASGI)

### 预期输出

```text
✅ fastmcp 已安装：3.4.7
✅ uvicorn 已安装：0.52.3
ℹ️  找到 docker 可执行文件：C:\Program Files\Docker\Docker\resources\bin\docker.EXE
   本课只展示 docker 命令，不调用引擎（无权限的机器上会报 access denied）。
CAN_RUN_ASGI = True
```

版本号会随 venv 升级而变、docker 的路径会随 Docker Desktop 安装位置而变 ——
**只有最后一行 `CAN_RUN_ASGI = True` 是必须为真的**：
它为假时第 2 节会自己降级成「打印一句中文说明」，不会抛异常。

## 1. 课案原版：Docker 部署（`10_部署_Docker.py`，48 行）

课案原版短得出人意料：**整个文件就是两个多行字符串 + 两次 `print`**。
因为「Docker 部署」这件事的知识量全在 Dockerfile 的语法里，而不在 Python 里 ——
我们要教的是**那几行 Dockerfile 每个字在干什么**，Python 只是搬运工。

先把原文原样跑出来看一眼。

### 1.1 课案原版的 Dockerfile 与 docker-compose.yml

In [ ]:
# 课案原文一字未改：这两个多行字符串，就是「要交给 docker build / docker compose 的两个文件」。
DOCKERFILE = """
FROM python:3.12-slim

WORKDIR /app

# 安装依赖（uv 更快；这里用 pip 演示，更通用）
COPY pyproject.toml uv.lock ./
RUN pip install --no-cache-dir uv && uv sync --frozen --no-dev

# 拷贝服务端代码
COPY 05_mcp/01_服务端.py ./

# HTTP 模式运行，监听 8000
EXPOSE 8000
CMD ["uv", "run", "python", "01_服务端.py", "http"]
"""

# 也可用 docker compose：
COMPOSE_YAML = """
services:
  mcp-life:
    build: .
    ports:
      - "8000:8000"
    restart: unless-stopped
"""

print("== Dockerfile ==")
print(DOCKERFILE)
print("== docker-compose.yml ==")
print(COMPOSE_YAML)

### 预期输出

```text
== Dockerfile ==

FROM python:3.12-slim

WORKDIR /app

# 安装依赖（uv 更快；这里用 pip 演示，更通用）
COPY pyproject.toml uv.lock ./
RUN pip install --no-cache-dir uv && uv sync --frozen --no-dev

# 拷贝服务端代码
COPY 05_mcp/01_服务端.py ./

# HTTP 模式运行，监听 8000
EXPOSE 8000
CMD ["uv", "run", "python", "01_服务端.py", "http"]

== docker-compose.yml ==

services:
  mcp-life:
    build: .
    ports:
      - "8000:8000"
    restart: unless-stopped
```

每个 `print` 后面那一串空行不是笔误：`DOCKERFILE` 这个字符串**本身以换行开头、以换行结尾**，
`print` 又会再加一个换行 —— 所以打印出来首尾各多一个空行。想看「干净版」，
用 `print(DOCKERFILE.strip())`。

### 1.2 Dockerfile 逐行讲解

| 指令 | 原文 | 在干什么 | 容易踩的坑 |
|---|---|---|---|
| `FROM` | `python:3.12-slim` | 选基础镜像。`slim` 去掉了编译工具链与文档，体积小很多 | 「需要编译的包」装不上；真遇到就换 `python:3.12` 完整版或多阶段构建 |
| `WORKDIR` | `/app` | 设容器内工作目录，后续 `COPY` / `RUN` / `CMD` 都在这里执行 | 等价于 `mkdir -p /app` 再 `cd /app`，目录不存在会自动建，不用自己 mkdir |
| `COPY` | `pyproject.toml uv.lock ./` | 只把锁文件复制进去 | 锁文件先于业务代码复制，是为了**让缓存失效范围最小**：改代码不会触发重装依赖 |
| `RUN` | `pip install ... uv && uv sync --frozen --no-dev` | 构建阶段装依赖，结果打进镜像层 | `--frozen` 表示严格按锁文件装，装不出来就报错（这正是我们要的确定性） |
| `COPY` | `05_mcp/01_服务端.py ./` | 复制服务端源码 | 注意是**仓库根相对路径** —— 所以 `docker build` 必须在仓库根执行，否则 `COPY` 找不到文件 |
| `EXPOSE` | `8000` | **声明**容器会监听 8000 | 它只是文档性质的声明，**真正对外暴露靠 `docker run -p 8000:8000`** |
| `CMD` | `["uv", "run", "python", "01_服务端.py", "http"]` | 容器启动时的默认命令 | 用 exec 数组写法（不是 shell 字符串），这样 Python 会成为 PID 1，能正确接收 `docker stop` 的 SIGTERM 并优雅退出 |

`docker-compose.yml` 则是把上面那条 `docker run` 变成声明式配置：`build: .` 相当于
`docker build`，`ports` 相当于 `-p`，`restart: unless-stopped` 让容器随 Docker 启动自动拉起
（除非你手动停过它）。所以 compose 版只需要一条命令：

````text
# 裸 docker：先构建，再运行
docker build -t mcp-life-service .
docker run -d -p 8000:8000 --name mcp-life mcp-life-service

# 或者 compose：一条命令搞定构建 + 运行 + 重启策略
docker compose up -d --build
````

| 片段 | 含义 |
|---|---|
| `docker build -t mcp-life-service .` | 用**当前目录**的 Dockerfile 构建，镜像打标签 `mcp-life-service` |
| `-d` | 后台运行（detach），不占着你的终端 |
| `-p 8000:8000` | 宿主机 8000 ←→ 容器 8000。**写反了就连不上** |
| `--name mcp-life` | 给容器起个固定名字，之后 `docker logs mcp-life` 就能看日志 |

客户端连接：`http://<服务器IP>:8000/mcp`，transport 选 `streamable-http`。

> ⚠️ **本 notebook 不会执行上面任何一条 docker 命令。** 本机 Docker 引擎当前账号无权限，
执行只会拿到 access denied —— 那不是「课案写错了」，而是环境限制。要真跑，
请把这段拿到有 Docker 权限的机器上执行；跑完 `docker ps` 能看到 `mcp-life` 就说明部署成功。

## 2. 完整版：开发环境 vs 生产环境（`11_部署_jxsd.py`）

课案完整版把「部署」拆成两种形态，并给出了**为什么生产要换写法**的答案：

| 形态 | 做法 | 适用 |
|---|---|---|
| 开发环境 | `python server.py` 直接跑（FastMCP 内置 uvicorn） | 快速开发、内部工具 |
| 生产环境 | `mcp.http_app()` 造 ASGI 应用 → uvicorn 多进程 | 正式部署、要自定义中间件 |

为什么生产要换？因为直接 `mcp.run()` 相当于「框架帮你起了一个 uvicorn」，你**控制不了它**：
加不了自己的中间件（CORS、限流、访问日志）、开不了多进程、也不方便挂到已有的 FastAPI 应用上。
`mcp.http_app()` 把 FastMCP 还原成一个**标准 Starlette/ASGI 应用**，
之后整个 ASGI 生态的能力就都能用了 —— 这也正是上一课 JWT 鉴权中间件挂得上去的原因。

本节的第 4 小节会**真的**把这套写法跑一次（不碰 Docker、只在本进程里起）。

### 2.1 开发环境：server.py（一行搞定）

最简形态：定义工具 → `mcp.run(transport="streamable-http", host=..., port=...)`。
课案原文如下，我们只把它打印出来 —— **不执行**，因为 `mcp.run()` 会一直阻塞，
notebook 里跑了它，这一格就永远不返回、后面所有格子都排不上队。

In [ ]:
# 课案原文（# pip install fastmcp 之后 `python server.py`）：
SERVER_PY = '''# server.py
from fastmcp import FastMCP


mcp = FastMCP("数学工具 🚀")


@mcp.tool
def add(a: float, b: float) -> float:
    """两数相加"""
    return a + b


@mcp.tool
def multiply(a: float, b: float) -> float:
    """两数相乘"""
    return a * b


if __name__ == "__main__":
    mcp.run(transport="streamable-http", host="0.0.0.0", port=8000)
'''

print("=" * 64)
print("① 开发环境：server.py")
print("=" * 64)
print(SERVER_PY)
print("运行：")
print("    python server.py")
print("    # 服务地址: http://localhost:8000/mcp/\n")

### 预期输出

```text
================================================================
① 开发环境：server.py
================================================================
# server.py
from fastmcp import FastMCP


mcp = FastMCP("数学工具 🚀")


@mcp.tool
def add(a: float, b: float) -> float:
    """两数相加"""
    return a + b


@mcp.tool
def multiply(a: float, b: float) -> float:
    """两数相乘"""
    return a * b


if __name__ == "__main__":
    mcp.run(transport="streamable-http", host="0.0.0.0", port=8000)

运行：
    python server.py
    # 服务地址: http://localhost:8000/mcp/
```

两个容易被忽略的点：

- `host="0.0.0.0"` 表示**监听所有网卡**。容器里必须这么写；写 `127.0.0.1` 的话端口只在
  容器内部回环，宿主机的 `-p` 映射根本连不上（Docker 新手最常见的坑）。本地单机调试
  建议改回 `127.0.0.1` 更安全。
- 把 `transport` 换成 `"stdio"` 就是**本地调试**形态：不监听端口，
  由调试客户端（或 IDE 的 MCP 插件、本课第 3 节的 Inspector）当子进程拉起。

### 2.2 生产环境：app.py —— 把 FastMCP 变成标准 ASGI 应用

关键只有一行：`app = mcp.http_app(transport="streamable-http")`。
返回的是一个**标准 Starlette/ASGI 应用**，所以 `uvicorn app:app` 才成立。

`app:app` 的含义是 **模块名 : 变量名** —— 「文件 `app.py` 里那个叫 `app` 的对象」。
uvicorn 会 import 这个模块再取属性，所以 `mcp.http_app()` 的返回值
**一定要赋给名为 `app` 的模块级变量**，改名就成了 `uvicorn app:server`。

In [ ]:
# 课案原文：
APP_PY = '''# app.py
from fastmcp import FastMCP


mcp = FastMCP("生产服务")


@mcp.tool
def process_data(input: str) -> str:
    """处理数据"""
    return f"已处理: {input}"


# 创建 ASGI 应用（默认使用 streamable-http 传输）
app = mcp.http_app(transport="streamable-http")
'''

print("=" * 64)
print("② 生产环境：app.py + uvicorn")
print("=" * 64)
print(APP_PY)
print("运行：")
print("    pip install uvicorn[standard]")
print("    uvicorn app:app --host 0.0.0.0 --port 8000                  # 单进程")
print("    uvicorn app:app --host 0.0.0.0 --port 8000 --workers 4      # 多进程（生产推荐）\n")

### 预期输出

```text
================================================================
② 生产环境：app.py + uvicorn
================================================================
# app.py
from fastmcp import FastMCP


mcp = FastMCP("生产服务")


@mcp.tool
def process_data(input: str) -> str:
    """处理数据"""
    return f"已处理: {input}"


# 创建 ASGI 应用（默认使用 streamable-http 传输）
app = mcp.http_app(transport="streamable-http")

运行：
    pip install uvicorn[standard]
    uvicorn app:app --host 0.0.0.0 --port 8000                  # 单进程
    uvicorn app:app --host 0.0.0.0 --port 8000 --workers 4      # 多进程（生产推荐）
```

### 2.3 生产 Dockerfile（课案原文）+ 多进程版本

和第 1 节的 Dockerfile 是同一件事的两种写法：第 1 节是「uv 项目 + 完整仓库」的构建方式，
这里是「单文件服务 + pip 直装」的最小方式。

> 变量名仍叫 `DOCKERFILE`（课案原文如此）—— 它和第 1 节**重名**，本节把它重新赋值覆盖掉了。
> 两块内容本来就来自两个不同的源文件，各自独立、互不依赖，只是课案取了同一个变量名。

In [ ]:
# 课案原文一笔未改。FROM / WORKDIR / COPY / RUN / EXPOSE / CMD 六行，逐行讲解见 markdown 表格。
DOCKERFILE = '''FROM python:3.13-slim

WORKDIR /app
COPY app.py .
RUN pip install fastmcp uvicorn

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

# 课案原文的 CMD 是**单进程**的。要开多进程（生产推荐），只改 CMD 一行：
DOCKERFILE_WORKERS = '''CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "4"]
'''

print("=" * 64)
print("③ Dockerfile（课案原文）")
print("=" * 64)
print(DOCKERFILE)
print("多进程版本只需改 CMD 一行：")
print(DOCKERFILE_WORKERS)
print("构建与运行：")
print("    docker build -t mcp-server .")
print("    docker run -d -p 8000:8000 mcp-server\n")

### 预期输出

```text
================================================================
③ Dockerfile（课案原文）
================================================================
FROM python:3.13-slim

WORKDIR /app
COPY app.py .
RUN pip install fastmcp uvicorn

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

多进程版本只需改 CMD 一行：
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "4"]

构建与运行：
    docker build -t mcp-server .
    docker run -d -p 8000:8000 mcp-server
```

| 指令 | 原文 | 在干什么 | 容易踩的坑 |
|---|---|---|---|
| `FROM` | `python:3.13-slim` | 基础镜像，slim 版体积小 | 需要编译的包装不上，真遇到换完整版 |
| `WORKDIR` | `/app` | 容器内工作目录 | 不存在会自动建 |
| `COPY` | `app.py .` | 只复制这一个文件 —— 服务本身就是单文件，容器里不需要整个项目 | 源文件必须在构建上下文里（`docker build` 的目录） |
| `RUN` | `pip install fastmcp uvicorn` | 构建阶段装依赖 | 生产上更推荐锁版本（`pip install "fastmcp==3.4.7"`）或 `COPY requirements.txt` 后 `pip install -r` |
| `EXPOSE` | `8000` | 声明性说明，不真正暴露 | 真正暴露靠 `docker run -p` |
| `CMD` | exec 数组 | 进程成为 PID 1，能优雅退出 | `--host 0.0.0.0` 是**必须**的，写 `127.0.0.1` 宿主机连不上 |

**多进程的隐藏陷阱**：MCP 服务端如果用「**进程内存态**」保存会话，
多进程下每个 worker 各存一份、互相看不见，表现为「刚连上就掉」。
出路有两条：改用 `stateless_http=True`，或把状态放到 Redis 这类外部存储里。

### 2.4 自检：真的把 ASGI 应用起一次（端口 8024）

上面全是「打印讲解」，那这套写法到底能不能跑？这一小节**真跑**：

1. 定义一个和 `app.py` 里**逐行对应**的最小服务端；
2. `mcp.http_app(transport="streamable-http")` 造出 ASGI 应用，**打印它的真实类型**；
3. 用 `uvicorn.Server` + 后台线程把它起在 `127.0.0.1:8024`（避开课案示例的 8000，
   也避开本章其它 notebook 用的 8100）；
4. 用 `fastmcp.Client` 连上去 `list_tools()` 再 `call_tool()` 一次；
5. 立刻 `should_exit` 关掉 —— **notebook 里不许留常驻服务**。

全程**不涉及 Docker**：Docker 只是把「这台机器上能跑的东西」打包，跑不跑得起来，
在宿主机上就能验证。

In [ ]:
import asyncio
import socket
import threading
import time

from fastmcp import FastMCP

HTTP_HOST = "127.0.0.1"
HTTP_PORT = 8024          # 生产/开发示例用 8000，本文件自检避开它
MCP_PATH = "/mcp"

# 和上面 APP_PY 里的写法完全一致，只是为了在本 notebook 里**真跑一次** mcp.http_app()。
mcp = FastMCP("生产服务")


@mcp.tool
def process_data(input: str) -> str:
    """处理数据"""
    return f"已处理: {input}"


# 端口探测与 01/02 里同理：连不上说明没人监听，不算错误。
def _port_open(host: str, port: int) -> bool:
    sock = socket.socket()
    sock.settimeout(0.5)
    try:
        sock.connect((host, port))
        return True
    except OSError:
        return False
    finally:
        sock.close()

上面这一格把 `_port_open` 定义好了；下面定义连上去调工具的那个协程。
源文件把 `from fastmcp import Client` 写在函数体里（只有真要连服务时才 import），
这里照抄 —— 这样一来「没装 fastmcp 时连 import 都不会发生」。

In [ ]:
# 用 01/02 同一套写法真起一次服务，再用 fastmcp.Client 连上去调一次工具做验证。
async def _probe(url: str) -> None:
    from fastmcp import Client

    async with Client(url) as client:
        tools = await client.list_tools()
        print(f"  可用工具：{[t.name for t in tools]}")
        result = await client.call_tool("process_data", {"input": "课案-生产环境示例"})
        print(f"  process_data 返回：{result.content[0].text}")


# ⚠️ notebook 与脚本的一处**硬差异**（本机实测踩到，见「常见坑」第 10 条）：
# 源文件在脚本里直接写 `asyncio.run(_probe(...))` 就行，但 notebook 的内核
# **本身就跑在一个事件循环里**，主线程再调 asyncio.run() 会立刻抛：
#     RuntimeError: asyncio.run() cannot be called from a running event loop
# 解法：把协程丢进一个**新线程**（新线程没有运行中的循环）再 asyncio.run()。
# 顺带把线程里的异常收集回主线程，避免「线程悄悄失败、notebook 却显示成功」。
def run_probe_in_thread(timeout: float = 60.0) -> None:
    box: list[BaseException] = []

    def _runner() -> None:
        try:
            asyncio.run(_probe(f"http://{HTTP_HOST}:{HTTP_PORT}{MCP_PATH}"))
        except BaseException as exc:      # noqa: BLE001 —— 收集起来回抛，别淹没在线程栈里
            box.append(exc)

    worker = threading.Thread(target=_runner, daemon=True)
    worker.start()
    worker.join(timeout=timeout)
    if box:
        raise box[0]

> **和源文件的一处差异**：源文件把 import 都堆在文件头；notebook 没有「文件头」，
> 所以本节把 `asyncio` / `socket` / `threading` / `time` 放在本小节第一格 ——
> 「哪一格用、哪一格 import」在 notebook 里比「统一堆在顶部」更好读。
> 函数体本身一字未改；**新增的只有上面这个 `run_probe_in_thread`**：它是
> 「notebook 内核已经在跑事件循环」这条环境差异的补丁，脚本里不需要它。

下面这一格是本节的重头戏：**起服务 → 连上去 → 调工具 → 关掉**。

括号里的 `8024` 端口如果已经被别的程序占了，这里会打印一句中文说明并跳过自检
（**不是报错**）；`fastmcp` / `uvicorn` 缺失时同理。两种降级都不会中断 notebook。

In [ ]:
print("=" * 64)
print(f"④ 自检：把 mcp.http_app() 造出来的 ASGI 应用真起一次（{HTTP_HOST}:{HTTP_PORT}）")
print("=" * 64)

if not CAN_RUN_ASGI:
    # 缺依赖就说清楚缺什么、怎么补，而不是让整本 notebook 崩在半路。
    print("⚠️  fastmcp / uvicorn 不可用，本自检未执行（请按「运行条件」把依赖装好再重跑这一格）。")
elif _port_open(HTTP_HOST, HTTP_PORT):
    print(f"ℹ️  {HTTP_HOST}:{HTTP_PORT} 已被占用，跳过自检（不影响上面的部署说明）。")
else:
    import uvicorn

    # app.py 里 `app = mcp.http_app(transport="streamable-http")` 就是这个对象
    app = mcp.http_app(transport="streamable-http")
    print(f"  app 类型：{type(app).__module__}.{type(app).__name__}   ← 标准 ASGI 应用")
    print(f"  等价启动命令：uvicorn <模块>:app --host 0.0.0.0 --port {HTTP_PORT}")

    config = uvicorn.Config(app, host=HTTP_HOST, port=HTTP_PORT, log_level="warning")
    # uvicorn.Server + 后台线程 + 轮询 started，与 01/02 完全同一套路。
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    # 轮询等就绪：uvicorn 真正开始监听后会把 server.started 置位，比写死 sleep 可靠。
    for _ in range(100):
        if server.started:
            break
        time.sleep(0.1)

    # 起来了才验证；没起来就直说「跳过自检」，不制造假成功。
    if server.started:
        # 注意这里调的是 run_probe_in_thread()（内部仍是 asyncio.run(_probe(...))），
        # 不是源文件里那句裸的 asyncio.run —— 原因见上面的注释与「常见坑」第 10 条。
        error = None
        try:
            run_probe_in_thread()
        except Exception as exc:      # noqa: BLE001 —— 自检失败要如实说，但不中断 notebook
            error = exc
        server.should_exit = True
        thread.join(timeout=10)
        if error is None:
            print("  ✅ ASGI 应用验证通过并已关闭（全程未使用 Docker）。")
        else:
            print(f"  ❌ 自检失败（{type(error).__name__}: {error}），上面的部署说明不受影响。")
    # 注意这一支也要显式报错：端口被占不是「验证通过」，不能让学员误判。
    else:
        print("  ❌ 端口无法监听，跳过自检。")

    print()
    print("小结：开发用 `python server.py` 一行搞定；生产用 app.py + uvicorn 多进程 + Docker。")

### 预期输出

```text
================================================================
④ 自检：把 mcp.http_app() 造出来的 ASGI 应用真起一次（127.0.0.1:8024）
================================================================
  app 类型：fastmcp.server.http.StarletteWithLifespan   ← 标准 ASGI 应用
  等价启动命令：uvicorn <模块>:app --host 0.0.0.0 --port 8024
  可用工具：['process_data']
  process_data 返回：已处理: 课案-生产环境示例
  ✅ ASGI 应用验证通过并已关闭（全程未使用 Docker）。

小结：开发用 `python server.py` 一行搞定；生产用 app.py + uvicorn 多进程 + Docker。
```

这一屏输出把课案里三句话都变成了**可验证的事实**：

| 输出 | 证明了什么 |
|---|---|
| `app 类型：fastmcp.server.http.StarletteWithLifespan` | `mcp.http_app()` 返回的就是标准 Starlette/ASGI 应用 —— 「`uvicorn app:app` 成立」「挂得上 CORS / 限流 / JWT 中间件」的技术前提 |
| `可用工具：['process_data']` | 用 `@mcp.tool` 注册的工具**真的**进了服务端的能力清单 |
| `process_data 返回：已处理: 课案-生产环境示例` | 课案那段 `app.py` 是**可运行的写法**，不是抄来跑不通的伪代码 |

> 端口被占 / 缺依赖时，这一格会换成一句中文说明（例如
> `ℹ️  127.0.0.1:8024 已被占用，跳过自检（不影响上面的部署说明）。`），
> 并且**不会**让后面的格子跑不下去。

## 3. MCP Inspector 调试工具（`12_调试工具_jxsd.py`）

MCP 服务端启动后是「哑」的 —— 它只在等人发 JSON-RPC，终端里看不到任何工具列表。
想确认「工具有没有注册上、参数 Schema 长什么样、调用会不会报错」，
写客户端代码太慢，**Inspector 点几下就行**。

它是 Anthropic 官方出品的 MCP **可视化调试工具**，通过 npm 生态分发。
三种手段的分工：

| 手段 | 适合场景 |
|---|---|
| `02_客户端_jxsd.py` 那种写代码的方式 | 可重复、可进 CI 的验证 |
| **MCP Inspector（本节）** | 手工探索：看看有哪些工具 / 资源 / 提示词，随手填参数试一次 |
| 打印日志 | 定位服务端内部逻辑问题 |

> ⚠️ **本 notebook 不会真的启动 Inspector**：`npx @modelcontextprotocol/inspector`
> 是**前台常驻**进程，跑在 notebook 里会把内核永久卡住。所以这里只**检查环境 + 打印命令**，
> 由你自己开一个终端窗口去跑。

> 📌 变量名沿用源文件：第 3 节的 `HTTP_HOST` / `HTTP_PORT` 会**覆盖**第 2 节同名的变量
> （8024 → 8100）。两块内容本来就来自两个不同的源文件，各自独立。

### 3.1 环境检查：`npx` / `node` 与「存在 ≠ 能用」

Inspector 通过 npm 生态分发，所以第一件事是确认 `npx` 在不在。

- 用 `shutil.which("npx")` 而不是直接 `subprocess.run(["npx", ...])`：
  Windows 上 npx 实际是 `npx.CMD` / `npx.ps1`，`shutil.which` 会按 `PATHEXT` 正确解析，
  直接跑 `"npx"` 反而可能 `FileNotFoundError`。
- `which` 只证明**文件在**，不证明**能用**（权限、损坏的 shim、PATH 里的同名目录都能骗过它），
  所以还要真跑一次 `npx --version`。
- `node` 也一起查：npx 只是「下载并拉起」，真正跑 Inspector 的是 node。

缺 `npx`（最常见的原因是没装 Node.js）时函数返回 `False`，让后面的步骤**整体降级为说明**，
而不是在半路抛 `FileNotFoundError` —— 学员该看到的是一句「去装 Node.js」，不是一堆栈。

In [ ]:
import subprocess
import shutil


def check_npx() -> bool:
    """检查本机有没有 npx（MCP Inspector 是通过 npm 生态分发的）。

    为什么要用 shutil.which 而不是直接 subprocess.run(["npx", ...])？
    Windows 上 npx 实际是 npx.CMD / npx.ps1，PATH 解析规则和 Linux 不同，
    shutil.which 会按 PATHEXT 正确找到可执行文件；直接跑 "npx" 反而可能 FileNotFoundError。
    """
    print("=" * 64)
    print("① 环境检查")
    print("=" * 64)

    # 两个都要查：npx 负责下载并拉起 Inspector，node 是它真正的运行时
    npx_path = shutil.which("npx")
    node_path = shutil.which("node")

    # 缺 npx 就直接返回 False，让后面步骤整体跳过；
    # 不能半路抛 FileNotFoundError —— 那样学员看到的是一堆栈，而不是「去装 Node.js」。
    if not npx_path:
        print("❌ 未找到 npx，Inspector 无法安装/启动。")
        print("   请先安装 Node.js（自带 npm / npx）：https://nodejs.org/")
        print("   或使用 nvm-windows / fnm 管理 Node 版本。")
        print("   安装后重开终端，再运行本文件确认。")
        return False

    print(f"✅ npx 已找到：{npx_path}")
    if node_path:
        print(f"✅ node 已找到：{node_path}")

    # 真跑一次 --version，确认它不只是「存在」而是「能用」
    # （which 只证明文件在，不证明它可执行：权限、损坏的 shim、PATH 里的同名目录都能骗过 which）
    try:
        result = subprocess.run(
            [npx_path, "--version"],
            capture_output=True,
            text=True,
            timeout=60,             # npx 首次可能触网，给足 60 秒
            encoding="utf-8",
            errors="replace",       # npm 输出偶尔含非 UTF-8 字节，替换掉比抛异常好
        )
        if result.returncode == 0:
            print(f"✅ npx 版本：{result.stdout.strip()}")
        else:
            # 版本号查不出来只警告、不算失败：npm 源慢或离线都不影响 Inspector 的可用性。
            print(f"⚠️  npx --version 返回码 {result.returncode}：{(result.stderr or '').strip()[:200]}")
    except subprocess.TimeoutExpired:
        print("⚠️  npx --version 超时（网络或 npm 源慢），但不影响后续使用。")
    except Exception as exc:
        print(f"⚠️  执行 npx --version 失败：{type(exc).__name__}: {exc}")

    return True

定义好了就真跑一次。这一格会**真的执行 `npx --version`** ——
首次运行如果 npm 源慢，可能要等十几秒，属于正常现象。

In [ ]:
has_npx = check_npx()

### 预期输出

```text
================================================================
① 环境检查
================================================================
✅ npx 已找到：F:\ProGramApp\nodejs\npx.CMD
✅ node 已找到：F:\ProGramApp\nodejs\node.EXE
✅ npx 版本：11.19.0
```

路径与版本随本机 Node 安装位置而变。**只有第一行「✅ npx 已找到」必须为真**：
它缺席时（只有 `❌ 未找到 npx，...` 那四行），后面的小节会给出「先装 Node.js」的指引，
notebook 依旧一路跑到底、不抛异常。

### 3.2 服务端在不在：Inspector「连不上」的头号原因

Inspector 自己不带服务端 —— 它只是个**客户端**。连不上时，
按经验排第一位的永远是「服务端根本没在跑」。

所以先探一下端口：**连不上不是错误**，只是「还没起服务」；
这里不抛异常，而是把补救命令打出来。

> 📌 课案原文这里探的是 `8000`（开发示例端口）。本套 notebook 的 MCP 服务端统一用
> **`8100`**（见 `01_服务端与客户端.ipynb` 的运行条件），所以这里同步改成 `8100` ——
> 避免和其它课程、其它服务撞端口，也避免「探到一个不相干的服务却以为是自己那个」。

In [ ]:
HTTP_HOST = "127.0.0.1"
HTTP_PORT = 8100
MCP_URL = f"http://{HTTP_HOST}:{HTTP_PORT}/mcp"


def check_server_running() -> bool:
    """看一眼 8100 端口有没有 MCP 服务在跑 —— Inspector 连不上多半是这个原因。"""
    sock = socket.socket()
    sock.settimeout(0.5)            # 探测用短超时：宁可在 0.5 秒内判定「没人监听」，也别卡住
    try:
        sock.connect((HTTP_HOST, HTTP_PORT))
        print(f"✅ 检测到 {MCP_URL} 已有服务在运行，可以直接用 Inspector 连它。")
        return True
    except OSError:
        # 连不上不是错误，只是「还没起服务」。这里不抛异常，而是把补救命令打出来
        print(f"ℹ️  {HTTP_HOST}:{HTTP_PORT} 当前没有服务。")
        print("   要用 Inspector 调试，请先另开一个窗口启动服务端：")
        print("       uv run Agent/05_mcp/01_服务端_jxsd.py http")
        return False
    finally:
        sock.close()                # 探测完立刻关闭，别把探测 socket 留成泄漏

In [ ]:
server_up = check_server_running()

### 预期输出

本机此刻没有 8100 服务时（**正常情况**，本课不要求你提前起服务）：

```text
ℹ️  127.0.0.1:8100 当前没有服务。
   要用 Inspector 调试，请先另开一个窗口启动服务端：
       uv run Agent/05_mcp/01_服务端_jxsd.py http
```

如果你先按 `01_服务端与客户端.ipynb` 起了服务端，这里会变成：

```text
✅ 检测到 http://127.0.0.1:8100/mcp 已有服务在运行，可以直接用 Inspector 连它。
```

两种都是「正常输出」：探测的结果只是用来决定后面打印哪一句建议。

### 3.3 课案的操作手册：安装与启动 / 连接服务端 / 功能使用

课案把 Inspector 的用法分四小节，下面这一格把其中**②③④**三节原样打印成终端里的操作手册
（分节标题就是课案的小节名）。四小节的内容先看一眼：

| 课案小节 | 内容要点 |
|---|---|
| ① 安装与启动 | `npx @modelcontextprotocol/inspector`，默认开 `http://localhost:6274` |
| ② 连接 MCP 服务端 | 三种传输：Streamable HTTP（推荐）/ SSE（已弃用）/ STDIO |
| ③ 功能使用 | Tools 列表、工具调用、Resources 浏览、Prompts 浏览 |
| ④ 调用示例 | 对着本套课案的具体文件怎么练（见下一小节的「这样练」清单） |

**为什么它是「前台常驻」进程**：Inspector 起了两个端口 —— `6274` 是浏览器 UI，
`6277` 是它内部的代理。关掉终端它就没，所以调试期间要单独占一个终端窗口。

In [ ]:
def print_guide() -> None:
    # —— 课案「安装与启动」 ——
    print()
    print("=" * 64)
    print("② 安装与启动（课案原文）")
    print("=" * 64)
    print("    npx @modelcontextprotocol/inspector")
    print()
    print("  首次运行会自动下载依赖；启动后终端会显示访问地址")
    print("  （默认 http://localhost:6274），用浏览器打开即可。")
    print()
    print("  常用变体：")
    print("    npx @modelcontextprotocol/inspector --version        # 看版本")
    print("    npx -y @modelcontextprotocol/inspector               # 跳过确认直接下载")
    print("    npx @modelcontextprotocol/inspector --help           # 看全部参数")
    print()
    # 先讲清「Inspector 是前台常驻进程」，再给命令 —— 否则学员会以为脚本卡住了。
    print("  ⚠️ 它是**前台常驻**进程：关掉终端 Inspector 就停了；")
    print("     调试期间请单开一个终端窗口跑它，别和别的命令挤在一起。")

    # —— 课案「连接 MCP 服务端」（三种传输） ——
    print()
    print("=" * 64)
    print("③ 连接 MCP 服务端（课案原文：支持三种传输方式，Streamable HTTP 推荐）")
    print("=" * 64)
    print("  在 Inspector 的连接页面里选传输方式，然后填地址：")
    print()
    # —— 传输方式对照表：和课案「四种协议」是同一套概念，只是换了个入口 ——
    print("    # | 传输方式 | Inspector 里怎么填 | 说明 |")
    print("    # |---|---|---|")
    print("    # | Streamable HTTP（推荐） | URL 填下面的地址 | 新项目一律选它 |")
    print("    # | SSE | URL 填 http://<主机>:<端口>/sse | 已弃用 |")
    print("    # | STDIO | Command / Args 分两格填 | 本地脚本，不占端口 |")
    print()
    print(f"    Streamable HTTP 的 URL： {MCP_URL}")
    print()
    # STDIO 方式：把「用哪个解释器 + 跑哪个脚本」告诉 Inspector，
    # 它就会像 02_客户端_jxsd.py 的 StdioTransport 一样把脚本拉成子进程
    print("    STDIO 方式（不用先起服务，Inspector 自己拉子进程）：")
    print(f"      Command : {sys.executable}")
    print(f"      Args    : {SERVER_SCRIPT} stdio")
    print("      （Args 里那个 stdio 参数不能省，否则会走自检分支去抢 8100 端口）")
    print()
    # —— 连不上时的排查清单，按「最可能的原因」排序 ——
    print("    连不上时按顺序排查：")
    print("      1. 服务端起了吗？（① 里的检测结果）")
    print("      2. URL 结尾的 /mcp 有没有漏？（漏了就 404）")
    print("      3. host 写的是 127.0.0.1 还是 0.0.0.0？跨机器访问要用实际 IP")
    print("      4. 带认证的服务要先拿令牌（见 08/09/10），Inspector 里在")
    print("         「Authentication」区选 Bearer Token 并粘贴 JWT")

    # —— 课案「功能使用」原表 ——
    # —— 配套本套课案怎么练：把 Inspector 的四个功能对到四个文件上 ——
    print()
    print("=" * 64)
    print("④ 功能使用（课案原表）")
    print("=" * 64)
    print("    # | 功能 | 说明 |")
    print("    # |---|---|")
    print("    # | Tools 列表 | 左侧面板展示服务端所有工具的名称和描述 |")
    print("    # | 工具调用 | 点击工具，填写参数表单，点击执行即可测试 |")
    print("    # | Resources 浏览 | 浏览服务端暴露的资源（文件、数据等） |")
    print("    # | Prompts 浏览 | 查看服务端提供的提示词模板 |")
    print()
    # 这段是「配套本套课案文件怎么练」——把 Inspector 的四个功能
    # 分别对上 01 / 03 / 04 / 09 四个文件，学员照着点一遍就全会了
    print("  对着本套课案文件可以这样练：")
    print("    · 连 01_服务端_jxsd.py  → Tools 里能看到 add / sub / mul / div，")
    print("      点 div 填 a=1,b=0 执行，能直接在界面上看到红字报错「除数不能为 0」")
    print("    · 连 03_资源_jxsd.py    → Resources 里固定资源 2 个、资源模板 4 个，")
    print("      点 users://top/3 就能读到数据库查出来的 TOP3")
    print("    · 连 04_提示词_jxsd.py  → Prompts 里填参数，右侧直接渲染出拼好的提示词")
    print("    · 连 09_权限_服务端_jxsd.py → 不填 Token 会连不上，填上 08 签发的 JWT 才通")
    print()
    # 收尾提醒：Inspector 解决「手工探索」，回归验证仍要写成代码（02 那种）。
    print("  调试完记得回来：Inspector 只解决「手工探索」，")
    print("  真正要固化下来的验证还是得写成 02_客户端_jxsd.py 那样的代码。")

> **和源文件的一处差异**：源文件里 `Args` 那一行写的是
> `__file__.replace('12_调试工具_jxsd.py', '01_服务端_jxsd.py')`，
> 靠脚本自己的路径推出服务端脚本的位置。**notebook 里没有 `__file__`**，
> 所以本节改用 `ROOT` 拼出 `SERVER_SCRIPT`，那一行改成直接打印它 ——
> 打印的仍是「解释器 + 服务端脚本 + stdio」这三样，只是路径由 notebook 推导。

In [ ]:
# notebook 里没有 __file__（模板第 6 节第 3 条）：改用**仓库根**推出服务端脚本的位置。
# 注意课案的 .py 已归档到 `Agent/_py_source/05_mcp/` —— 所以这里指向归档后的**真实文件**，
# 这样照「预期输出」里那两行填进 Inspector 的 STDIO 方式，是**可以直接用的**。
SERVER_SCRIPT = ROOT / "Agent" / "_py_source" / "05_mcp" / "01_服务端_jxsd.py"

print_guide()

### 预期输出

```text

================================================================
② 安装与启动（课案原文）
================================================================
    npx @modelcontextprotocol/inspector

  首次运行会自动下载依赖；启动后终端会显示访问地址
  （默认 http://localhost:6274），用浏览器打开即可。

  常用变体：
    npx @modelcontextprotocol/inspector --version        # 看版本
    npx -y @modelcontextprotocol/inspector               # 跳过确认直接下载
    npx @modelcontextprotocol/inspector --help           # 看全部参数

  ⚠️ 它是**前台常驻**进程：关掉终端 Inspector 就停了；
     调试期间请单开一个终端窗口跑它，别和别的命令挤在一起。

================================================================
③ 连接 MCP 服务端（课案原文：支持三种传输方式，Streamable HTTP 推荐）
================================================================
  在 Inspector 的连接页面里选传输方式，然后填地址：

    # | 传输方式 | Inspector 里怎么填 | 说明 |
    # |---|---|---|
    # | Streamable HTTP（推荐） | URL 填下面的地址 | 新项目一律选它 |
    # | SSE | URL 填 http://<主机>:<端口>/sse | 已弃用 |
    # | STDIO | Command / Args 分两格填 | 本地脚本，不占端口 |

    Streamable HTTP 的 URL： http://127.0.0.1:8100/mcp

    STDIO 方式（不用先起服务，Inspector 自己拉子进程）：
      Command : F:\ProGram\Python_Base\.venv\Scripts\python.exe
      Args    : F:\ProGram\Python_Base\Agent\_py_source\05_mcp\01_服务端_jxsd.py stdio
      （Args 里那个 stdio 参数不能省，否则会走自检分支去抢 8100 端口）

    连不上时按顺序排查：
      1. 服务端起了吗？（① 里的检测结果）
      2. URL 结尾的 /mcp 有没有漏？（漏了就 404）
      3. host 写的是 127.0.0.1 还是 0.0.0.0？跨机器访问要用实际 IP
      4. 带认证的服务要先拿令牌（见 08/09/10），Inspector 里在
         「Authentication」区选 Bearer Token 并粘贴 JWT

================================================================
④ 功能使用（课案原表）
================================================================
    # | 功能 | 说明 |
    # |---|---|
    # | Tools 列表 | 左侧面板展示服务端所有工具的名称和描述 |
    # | 工具调用 | 点击工具，填写参数表单，点击执行即可测试 |
    # | Resources 浏览 | 浏览服务端暴露的资源（文件、数据等） |
    # | Prompts 浏览 | 查看服务端提供的提示词模板 |

  对着本套课案文件可以这样练：
    · 连 01_服务端_jxsd.py  → Tools 里能看到 add / sub / mul / div，
      点 div 填 a=1,b=0 执行，能直接在界面上看到红字报错「除数不能为 0」
    · 连 03_资源_jxsd.py    → Resources 里固定资源 2 个、资源模板 4 个，
      点 users://top/3 就能读到数据库查出来的 TOP3
    · 连 04_提示词_jxsd.py  → Prompts 里填参数，右侧直接渲染出拼好的提示词
    · 连 09_权限_服务端_jxsd.py → 不填 Token 会连不上，填上 08 签发的 JWT 才通

  调试完记得回来：Inspector 只解决「手工探索」，
  真正要固化下来的验证还是得写成 02_客户端_jxsd.py 那样的代码。
```

`Command` 之所以是 `sys.executable`（当前内核用的那个解释器），是因为 STDIO 方式下
**Inspector 要自己把服务端脚本拉成子进程** —— 解释器选错（比如落到 Windows Store 占位符
`python.exe`）就会静默失败，所以这里显式用「跑 notebook 的这个解释器」最稳。
上面 `Command` / `Args` 两行的绝对路径**随你把仓库放在哪而变**，
关键是它们指向「当前解释器」与「课案的服务端脚本」。

> 📌 上面 ②③ 里那条 `uv run Agent/05_mcp/01_服务端_jxsd.py http` 是**课案原文**，
> 保留不改；但课案的 `.py` 现在已经归档到 `Agent/_py_source/05_mcp/`，
> 在本仓库里照着敲要把路径换成 `Agent/_py_source/05_mcp/01_服务端_jxsd.py`
> （或者直接跑本章的 notebook `01_服务端与客户端.ipynb`）。

### 3.4 一键复现命令：把上面两张表变成两条命令

手抄容易抄错，所以最后把「两个终端 + 浏览器」的完整复现路径打出来。
注意下面两个分支都**只用 `if` 判断、不 `sys.exit`**：源文件跑在脚本里，
没环境就 `sys.exit(0)` 结束；notebook 里退出会杀掉内核，所以改成「打印对应的下一步」。

In [ ]:
print()
print("=" * 64)
print("⑤ 一键复现命令")
print("=" * 64)
print("    终端 1（服务端）：")
print("        uv run Agent/05_mcp/01_服务端_jxsd.py http")
print("    终端 2（Inspector）：")
print("        npx @modelcontextprotocol/inspector")
print("    浏览器： http://localhost:6274  →  传输选 Streamable HTTP")
print(f"            URL 填 {MCP_URL}")
print()

# 源文件这两个分支都用 sys.exit(0)：缺环境是「学员的机器还没配好」，不是脚本出错。
# notebook 里不能退内核，所以改成「各自打印一句下一步」—— 三种情况互不排斥，用三个 if 即可。
if not has_npx:
    print("⚠️  本机缺 npx，上面这两步请先装 Node.js 再执行。")

if not server_up:
    print("ℹ️  服务端当前没起，照上面「终端 1」的命令起来之后再连。")

if has_npx and server_up:
    print("✅ 环境就绪：npx 可用 + 服务端在跑，直接执行「终端 2」那条命令即可开始调试。")

### 预期输出

本机此刻的实际情况（有 npx、没有 8100 服务）：

```text

================================================================
⑤ 一键复现命令
================================================================
    终端 1（服务端）：
        uv run Agent/05_mcp/01_服务端_jxsd.py http
    终端 2（Inspector）：
        npx @modelcontextprotocol/inspector
    浏览器： http://localhost:6274  →  传输选 Streamable HTTP
            URL 填 http://127.0.0.1:8100/mcp

ℹ️  服务端当前没起，照上面「终端 1」的命令起来之后再连。
```

三种结尾分支对照（**都算正常**，取决于你的机器状态）：

| 条件 | 结尾那句 | 下一步 |
|---|---|---|
| 缺 npx | `⚠️  本机缺 npx，上面这两步请先装 Node.js 再执行。` | 装 Node.js |
| 有 npx、服务端没起 | `ℹ️  服务端当前没起，照上面「终端 1」的命令起来之后再连。` | 先起服务端 |
| 有 npx、服务端在跑 | `✅ 环境就绪：npx 可用 + 服务端在跑，直接执行「终端 2」那条命令即可开始调试。` | 直接开 Inspector |

## 小结

- **开发 vs 生产**：`mcp.run()` 是「框架替你起 uvicorn」，够用但不可控；
  `mcp.http_app()` 把 FastMCP 还原成**标准 ASGI 应用**，从此中间件、多进程、
  挂进已有 FastAPI 应用都成立 —— 这是上一课 JWT 鉴权能挂上去的前提。
- **`uvicorn app:app`** 里的 `app:app` 是 **模块名 : 变量名**，
  所以 `http_app()` 的返回值必须赋给模块级变量 `app`。
- **Dockerfile 的六行**：`FROM` 选底、`WORKDIR` 定目录、`COPY` 放文件、
  `RUN` 装依赖、`EXPOSE` 声明端口（**真正暴露靠 `-p`**）、`CMD` 定启动命令
  （`--host 0.0.0.0` 是必须的）。
- **`docker run -d -p 8000:8000`**：`-d` 后台、`-p 宿主机:容器`；compose 只是把它声明化。
- **本课真的验证了什么**：第 2 节在进程内真起了一次 ASGI 应用、真连上去列出了工具并调用成功
  —— 「课案原文能跑」这件事是**跑出来的**，不是声称的。
- **Inspector 只解决「手工探索」**：看一眼有哪些工具、参数 Schema 长什么样、随手试一次。
  要固化下来的回归验证，仍然得写成 `02_客户端_jxsd.py` 那样的代码。
- **本课不做的两件事**：执行 docker 命令（本机无权限）、启动 Inspector（前台常驻会卡内核）。
  这两件事都只以「命令 + 说明」的形式出现。

## 常见坑

1. **`--host 0.0.0.0` 与 `127.0.0.1` 选错**：容器里、跨机器访问一律要 `0.0.0.0`；
   写 `127.0.0.1` 时端口只在容器内部回环，宿主机的 `-p` 映射**连不上**。
2. **`EXPOSE` 不等于暴露端口**：它只是声明。真正对外暴露靠 `docker run -p 8000:8000`。
3. **`-p` 写反**（`-p 8000:80`）会让宿主机端口映射到一个容器里没人监听的端口上，表现为「连不上」。
4. **`COPY` 路径是相对构建上下文的**：`COPY 05_mcp/01_服务端.py ./` 要求 `docker build` 在仓库根执行，
   换目录执行就找不到文件。
5. **多进程 + 内存态会话会「刚连上就掉」**：每个 worker 各存一份会话，互相看不见。
   改用 `stateless_http=True`，或把状态放到 Redis 这类外部存储。
6. **`uvicorn app:app` 里的变量名不能改**：`http_app()` 的返回值不叫 `app` 的话，
   启动命令得跟着改成 `模块名:那个变量名`，否则 `AttributeError`。
7. **Inspector 连不上的排查顺序**：先看服务端在不在 → 再看 URL 结尾的 `/mcp` 有没有漏（漏了 404）
   → 再看 host 用的是不是对方能访问到的地址 → 最后才看认证（带 JWT 的服务要先拿令牌，
   在 Inspector 的「Authentication」区选 Bearer Token）。
8. **别在 notebook / 脚本里直接跑 `npx @modelcontextprotocol/inspector`**：它是前台常驻进程，
   会把内核（或终端）卡住。要跑就单开一个终端窗口。
9. **`mcp.run(...)` 同理**：它会一直阻塞。notebook 里想「起服务」，
   用第 2 节那套 `uvicorn.Server` + 后台线程 + 轮询 `server.started`，
   并在末尾 `should_exit = True` 把它收掉。
10. **notebook 里不能直接 `asyncio.run(...)`**（本课实测踩到）：
    Jupyter 内核**本身就在一个事件循环里**，主线程再调 `asyncio.run()` 会立刻抛
    `RuntimeError: asyncio.run() cannot be called from a running event loop`。
    同一个文件在 `python xxx.py` 下跑得好好的，搬进 notebook 就炸 —— 就是这里。
    解法是把协程交给一个**新线程**（新线程没有运行中的循环）再 `asyncio.run()`，
    见 2.4 节的 `run_probe_in_thread`。顺带提醒：别用「顶层 `await`」绕，
    `ast.parse` 不认它，`nbtool.py check` 会直接判语法错误。

## 官方链接

- FastMCP HTTP 部署（`http_app`、uvicorn、Dockerfile 示例）：<https://gofastmcp.com/deployment/http>
- MCP Inspector 官方文档（安装、三种传输、功能面板）：<https://modelcontextprotocol.io/docs/tools/inspector>
- Inspector 源码与 issue：<https://github.com/modelcontextprotocol/inspector>
- MCP 协议规范（transport 一节，讲清 streamable-http / sse / stdio 的区别）：<https://modelcontextprotocol.io/specification/2025-06-18/basic/transports>